# 03 · Claude + a code-interpreter tool

Expose sandbox execution to Claude as a **tool**. The model decides when to call
`execute_python`; you run the code in a RaitBox sandbox and feed the result back
until it answers — the standard tool-use loop, over **Bedrock's Converse API**.

**Concepts:** tool use / function calling; the sandbox is the tool's execution
backend; `toolUse` → run → `toolResult` → repeat. Real model, no stubs.

**Prerequisites:** AWS credentials + Bedrock model access.

In [ ]:
%pip install -q raitbox

### Configuration & tool schema
Bedrock client and the `execute_python` tool (name + JSON input schema).

In [ ]:
import os

import boto3

from raitbox import Sandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("RAITBOX_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

SYSTEM = [{"text": "You are a careful analyst. Use the execute_python tool to compute "
           "answers from data instead of estimating, then state the result plainly."}]

TOOL_CONFIG = {"tools": [{"toolSpec": {
    "name": "execute_python",
    "description": "Run Python in a stateful sandbox kernel; returns stdout/stderr/errors.",
    "inputSchema": {"json": {"type": "object",
        "properties": {"code": {"type": "string", "description": "Python for one cell."}},
        "required": ["code"]}},
}}]}

### The tool + tool-use loop
`run_tool` executes code; `chat` loops model↔sandbox until Claude stops calling the tool.

In [ ]:
def run_tool(sbx, code):
    ex = sbx.run_code(code, on_stdout=lambda o: print("[py]", o, end=""))
    return f"ERROR {ex.error.name}: {ex.error.value}" if ex.error else ("".join(ex.logs.stdout) or "(ok)")


def chat(sbx, question):
    messages = [{"role": "user", "content": [{"text": question}]}]
    while True:
        resp = bedrock.converse(modelId=MODEL_ID, system=SYSTEM, messages=messages,
                                toolConfig=TOOL_CONFIG, inferenceConfig={"maxTokens": 4096, "temperature": 0})
        out = resp["output"]["message"]
        messages.append(out)
        if resp["stopReason"] != "tool_use":
            return out
        results = [{"toolResult": {"toolUseId": b["toolUse"]["toolUseId"],
                    "content": [{"text": run_tool(sbx, b["toolUse"]["input"]["code"])}]}}
                   for b in out["content"] if "toolUse" in b]
        messages.append({"role": "user", "content": results})

### Run it
Ask a question that needs real computation over inline data.

In [ ]:
DATA = [12, 15, 14, 10, 92, 13, 11, 16, 14, 9, 88, 15]

with Sandbox.create(region_name=REGION) as sbx:
    sbx.run_code(f"samples = {DATA}")
    final = chat(sbx, "Given the list `samples` already defined in the kernel, compute the mean "
                      "and standard deviation, then list the outliers more than 2 std devs from the mean.")

for b in final["content"]:
    if "text" in b:
        print("\nClaude:", b["text"])

### Recap & next
Same idea inside a framework: `05-langgraph-bedrock-code-interpreter`. Or produce an artifact: `04-data-analysis-chart`.